In [1]:
import pandas as pd

# Seulement les colonnes utiles : la population complète tient alors en mémoire
colonnes = ["idmutation", "anneemut", "l_codinsee", "libtypbien", "valeurfonc", "sbati"]

pop = pd.read_parquet("../data/dvf_plus_residentiel/", columns=colonnes)

print(pop.shape)
pop.head()

(9680495, 6)


,idmutation,anneemut,l_codinsee,libtypbien,valeurfonc,sbati
0,16,2014,01378,UNE MAISON,222300.0,88.0
1,476,2014,01160,UNE MAISON,1084000.0,212.0
2,8975,2014,01022,UNE MAISON,139000.0,90.0
3,9113,2014,01210,UN APPARTEMENT,75700.0,41.0
4,9498,2014,01185,UNE MAISON,58000.0,39.0


In [2]:
# Prix au m² de chaque vente
pop["prix_m2"] = pop["valeurfonc"] / pop["sbati"]

# Les trois règles d'exclusion de l'EDA
avant = len(pop)
pop = pop[
    pop["valeurfonc"].notna()
    & (pop["valeurfonc"] > 0)
    & (pop["prix_m2"] >= 50)
    & (pop["prix_m2"] <= 50000)
]
apres = len(pop)
print(f"Avant : {avant:,} | Après : {apres:,} | Exclues : {avant - apres:,} ({(avant - apres) / avant:.2%})")

# Type de bien en version courte, pour les noms de colonnes
pop["type_bien"] = pop["libtypbien"].map({"UNE MAISON": "maison", "UN APPARTEMENT": "appart"})
print(pop["type_bien"].value_counts(dropna=False))

Avant : 9,680,495 | Après : 9,655,421 | Exclues : 25,074 (0.26%)
type_bien
maison    5523372
appart    4132049
Name: count, dtype: int64


In [3]:
# 1. Par type de bien : une ligne par commune × année × type
par_type = (
    pop.groupby(["l_codinsee", "anneemut", "type_bien"])["prix_m2"]
    .agg(n="count", prix_m2_moy="mean", prix_m2_med="median")
    .unstack("type_bien")          # le type passe en colonnes : une ligne par commune × année
)
# Noms de colonnes : ("n", "maison") devient "n_maison"
par_type.columns = [f"{stat}_{typ}" for stat, typ in par_type.columns]

# 2. Tous types confondus
ensemble = (
    pop.groupby(["l_codinsee", "anneemut"])["prix_m2"]
    .agg(n_ensemble="count", prix_m2_moy_ensemble="mean", prix_m2_med_ensemble="median")
)

# 3. Assemblage
cible = par_type.join(ensemble).reset_index()
cible = cible.rename(columns={"l_codinsee": "code_commune", "anneemut": "annee"})

# Une commune sans vente d'un type donne n = NaN : on le remplace par 0
for col in ["n_maison", "n_appart"]:
    cible[col] = cible[col].fillna(0).astype(int)

# Prix arrondis à l'euro
cols_prix = [c for c in cible.columns if c.startswith("prix_m2")]
cible[cols_prix] = cible[cols_prix].round(0)

print(cible.shape)
cible.head()

(367779, 11)


,code_commune,annee,n_appart,n_maison,prix_m2_moy_appart,prix_m2_moy_maison,prix_m2_med_appart,prix_m2_med_maison,n_ensemble,prix_m2_moy_ensemble,prix_m2_med_ensemble
0,01001,2014,0,10,NaN,1709.0,NaN,1893.0,10,1709.0,1893.0
1,01001,2015,0,10,NaN,1729.0,NaN,1782.0,10,1729.0,1782.0
2,01001,2016,0,7,NaN,1995.0,NaN,1807.0,7,1995.0,1807.0
3,01001,2017,0,14,NaN,1826.0,NaN,1827.0,14,1826.0,1827.0
4,01001,2018,0,9,NaN,1906.0,NaN,1933.0,9,1906.0,1933.0


In [4]:
# Une ligne par commune × année × type, puis le type passe en colonnes
cible = (
    pop.groupby(["l_codinsee", "anneemut", "type_bien"])["prix_m2"]
    .agg(n="count", prix_m2_med="median")
    .unstack("type_bien")
)
# Noms de colonnes : ("n", "maison") devient "n_maison"
cible.columns = [f"{stat}_{typ}" for stat, typ in cible.columns]
cible = cible.reset_index().rename(columns={"l_codinsee": "code_commune", "anneemut": "annee"})

# Une commune sans vente d'un type : n = 0 (le prix reste vide)
for col in ["n_maison", "n_appart"]:
    cible[col] = cible[col].fillna(0).astype(int)

# Prix arrondis à l'euro
cible[["prix_m2_med_appart", "prix_m2_med_maison"]] = cible[["prix_m2_med_appart", "prix_m2_med_maison"]].round(0)

# Les six colonnes, dans l'ordre choisi
cible = cible[["code_commune", "annee", "n_appart", "n_maison", "prix_m2_med_appart", "prix_m2_med_maison"]]

print(cible.shape)
print("Total des ventes :", cible["n_appart"].sum() + cible["n_maison"].sum())   # doit valoir 9 655 421
cible.head()

(367779, 6)
Total des ventes : 9655421


,code_commune,annee,n_appart,n_maison,prix_m2_med_appart,prix_m2_med_maison
0,01001,2014,0,10,NaN,1893.0
1,01001,2015,0,10,NaN,1782.0
2,01001,2016,0,7,NaN,1807.0
3,01001,2017,0,14,NaN,1827.0
4,01001,2018,0,9,NaN,1933.0


In [5]:
import os

chemin_csv = "../data/processed/cible_commune_annee_type.csv"
chemin_parquet = "../data/processed/cible_commune_annee_type.parquet"

cible.to_csv(chemin_csv, index=False)
cible.to_parquet(chemin_parquet, index=False)

# Taille des deux fichiers
print(f"CSV     : {os.path.getsize(chemin_csv) / 1e6:.1f} Mo")
print(f"Parquet : {os.path.getsize(chemin_parquet) / 1e6:.1f} Mo")

# Relecture : le code commune garde-t-il son zéro ?
print("CSV sans précaution :", pd.read_csv(chemin_csv)["code_commune"].head(3).tolist())
print("CSV avec dtype      :", pd.read_csv(chemin_csv, dtype={"code_commune": str})["code_commune"].head(3).tolist())
print("Parquet             :", pd.read_parquet(chemin_parquet)["code_commune"].head(3).tolist())

CSV     : 9.0 Mo
Parquet : 1.7 Mo
CSV sans précaution : ['01001', '01001', '01001']
CSV avec dtype      : ['01001', '01001', '01001']
Parquet             : ['01001', '01001', '01001']


/tmp/ipykernel_30389/1201731880.py:14: DtypeWarning: Columns (0: code_commune) have mixed types. Specify dtype option on import or set low_memory=False.
  print("CSV sans précaution :", pd.read_csv(chemin_csv)["code_commune"].head(3).tolist())
